<a href='https://colab.research.google.com/github/Emelecto/QuantLab/blob/main/web/content/cursos/fundamentos/notebooks/c1_l1_edge.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg'/></a>

# C1-L1 · Edge y expectativa
Simula un juego con edge conocido y mide su expectativa. Solo librería estándar.

In [ ]:
import csv, math, random, urllib.request
from pathlib import Path

CSV = 'c1_l1_moneda.csv'
URL = 'https://raw.githubusercontent.com/Emelecto/QuantLab/main/web/content/cursos/fundamentos/data/' + CSV
csv_path = Path(CSV)
try:
    csv_path.write_bytes(urllib.request.urlopen(URL, timeout=15).read())
    print('Fuente: URL (Colab)')
except Exception as e:
    print('Sin red, uso fallback local:', e)
    for cand in [Path('../data') / CSV, Path('data') / CSV, Path(CSV)]:
        if cand.exists():
            csv_path = cand
            break
print('CSV:', csv_path.resolve())

In [ ]:
wins, total, pnl_total = 0, 0, 0
with open(csv_path, newline='', encoding='utf-8') as f:
    for row in csv.DictReader(f):
        total += 1
        wins += int(row['win'])
        pnl_total += float(row['pnl'])
p_hat = wins / total
ev_empirica = pnl_total / total
ev_teorica = 0.4 * 2 - 0.6 * 1
print(f'jugadas={total} wins={wins} p_hat={p_hat:.3f}')
print(f'expectativa empírica={ev_empirica:+.3f} €/jugada  teórica={ev_teorica:+.3f}')

## Experimento: ¿y si el juego fuera malo?
Cambia `p_ganar` a 0.3 y repite 500 jugadas simuladas.

In [ ]:
def simular(p_ganar, pago=2, perdida=1, n=500, seed=1):
    rng = random.Random(seed)
    eq, curve = 0.0, []
    for _ in range(n):
        eq += pago if rng.random() < p_ganar else -perdida
        curve.append(eq)
    return curve

bueno = simular(0.4)
malo = simular(0.3)
print(f'juego bueno (p=0.4): PnL final={bueno[-1]:+.1f}')
print(f'juego malo  (p=0.3): PnL final={malo[-1]:+.1f}')
print('Conclusión: con E<0, más repeticiones = más pérdida. Con E>0, el ruido se compensa.')

In [ ]:
# Chequeo automático
assert abs(ev_teorica - 0.2) < 1e-9, 'la expectativa teórica debe ser +0.20'
assert bueno[-1] > 0 > malo[-1], 'el juego bueno debe terminar arriba y el malo abajo (seed fija)'
print('OK: expectativa verificada')